In [4]:
# Celda 1: imports y rutas
from pathlib import Path
import json
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt

# Rutas base (ajústalas si tu notebook está en otro lado)
CAPSTONE = Path.cwd()  # carpeta actual
DATOS    = CAPSTONE / "DATOS"

ORIG_ROOT = DATOS / "01-openneuro"                        # OpenNeuro original
DERIV_DIR = DATOS / "02-derived-ecg-dataset" / "datos"    # tu dataset derivado

INDEX_CSV = DERIV_DIR / "index_master.csv"

print("ORIG_ROOT:", ORIG_ROOT.resolve())
print("DERIV_DIR:", DERIV_DIR.resolve())
print("INDEX_CSV:", INDEX_CSV.resolve())


ORIG_ROOT: /workspaces/CAPSTONE-SEGUIMIENTO/DATOS/01-openneuro
DERIV_DIR: /workspaces/CAPSTONE-SEGUIMIENTO/DATOS/02-derived-ecg-dataset/datos
INDEX_CSV: /workspaces/CAPSTONE-SEGUIMIENTO/DATOS/02-derived-ecg-dataset/datos/index_master.csv


In [5]:
# Celda 2: cargar el índice maestro y elegir ejemplo
df = pd.read_csv(INDEX_CSV)
print("Total segmentos:", len(df))
display(df.head(10))

# Elige una fila a verificar (puedes cambiar el índice o filtrar por sujeto/run)
row_idx = 0  # <-- cámbialo si quieres otro
row = df.iloc[row_idx].to_dict()

print("\nFila elegida:")
for k in ["segment_id","subject","session","run","event_type","onset_sec","duration_sec","cut_start_sec","cut_end_sec","fs_hz"]:
    print(f"{k}: {row.get(k)}")

# Rutas originales desde el índice (guardadas por el builder)
src_ecg_edf   = Path(row["source_ecg"])
src_events_tsv = Path(row["source_events"])
dst_npz       = Path(row["dst_npz"])
dst_json      = Path(row["dst_json"])

print("\nARCHIVOS:")
print("ECG EDF:", src_ecg_edf)
print("EEG events TSV:", src_events_tsv)
print("Segmento derivado NPZ:", dst_npz)
print("Segmento metadata JSON:", dst_json)


Total segmentos: 27


,segment_id,subject,session,task,run,source_ecg,source_events,event_index,event_type,onset_sec,duration_sec,cut_start_sec,cut_end_sec,pre_sec,post_sec,n_samples,fs_hz,dst_npz,dst_json
0,sub-001_ses-01_task-szMonitoring_run-03_seg-00,1,1,szMonitoring,3,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,0,sz_foc_ia_nm,57975.0,72.0,57075.0,58947.0,900.0,900.0,479232,256.0,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...
1,sub-001_ses-01_task-szMonitoring_run-05_seg-00,1,1,szMonitoring,5,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,0,sz_foc_a_nm,60968.0,81.0,60068.0,61949.0,900.0,900.0,481536,256.0,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...
2,sub-001_ses-01_task-szMonitoring_run-07_seg-00,1,1,szMonitoring,7,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,0,sz_foc_ia_m_tonic,16300.0,143.0,15400.0,17343.0,900.0,900.0,497408,256.0,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...
3,sub-001_ses-01_task-szMonitoring_run-08_seg-00,1,1,szMonitoring,8,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,0,sz_foc_a_m_automatisms,28285.0,25.0,27385.0,29210.0,900.0,900.0,467200,256.0,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...
4,sub-002_ses-01_task-szMonitoring_run-01_seg-00,2,1,szMonitoring,1,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,0,sz_foc_a_nm,4046.0,20.0,3146.0,4966.0,900.0,900.0,465920,256.0,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...
5,sub-002_ses-01_task-szMonitoring_run-01_seg-01,2,1,szMonitoring,1,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,1,sz_foc_ia_m_hyperkinetic,10629.0,181.0,9729.0,11380.0,900.0,900.0,422656,256.0,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...
6,sub-002_ses-01_task-szMonitoring_run-02_seg-00,2,1,szMonitoring,2,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,0,sz_foc_a_m_hyperkinetic,5224.0,112.0,4324.0,6236.0,900.0,900.0,489472,256.0,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...
7,sub-002_ses-01_task-szMonitoring_run-02_seg-01,2,1,szMonitoring,2,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,1,sz_foc_ia_m_hyperkinetic,13745.0,180.0,12845.0,14825.0,900.0,900.0,506880,256.0,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...
8,sub-002_ses-01_task-szMonitoring_run-03_seg-00,2,1,szMonitoring,3,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,0,sz_foc_ia_m_hyperkinetic,31537.0,52.0,30637.0,32489.0,900.0,900.0,474112,256.0,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...
9,sub-002_ses-01_task-szMonitoring_run-05_seg-00,2,1,szMonitoring,5,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,/Users/isaurac/Desktop/Capstone/DATOS/01-openn...,0,sz_foc_ia_m_hyperkinetic,69796.0,153.0,68896.0,70849.0,900.0,900.0,499968,256.0,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...,/Users/isaurac/Desktop/Capstone/DATOS/02-deriv...



Fila elegida:
segment_id: sub-001_ses-01_task-szMonitoring_run-03_seg-00
subject: 1
session: 1
run: 3
event_type: sz_foc_ia_nm
onset_sec: 57975.0
duration_sec: 72.0
cut_start_sec: 57075.0
cut_end_sec: 58947.0
fs_hz: 256.0

ARCHIVOS:
ECG EDF: /Users/isaurac/Desktop/Capstone/DATOS/01-openneuro/sub-001/ses-01/ecg/sub-001_ses-01_task-szMonitoring_run-03_ecg.edf
EEG events TSV: /Users/isaurac/Desktop/Capstone/DATOS/01-openneuro/sub-001/ses-01/eeg/sub-001_ses-01_task-szMonitoring_run-03_events.tsv
Segmento derivado NPZ: /Users/isaurac/Desktop/Capstone/DATOS/02-derived-ecg-dataset/datos/sub-001/ses-01/run-03/sub-001_ses-01_task-szMonitoring_run-03_seg-00.npz
Segmento metadata JSON: /Users/isaurac/Desktop/Capstone/DATOS/02-derived-ecg-dataset/datos/sub-001/ses-01/run-03/sub-001_ses-01_task-szMonitoring_run-03_seg-00.json


In [ ]:
# Celda 3: leer eventos y ECG original
# (1) Eventos del EEG (para ver onset/duration y tipo)
ev = pd.read_csv(src_events_tsv, sep="\t")
print("Eventos en el TSV original:", len(ev))
display(ev.head())

# Intentar detectar columnas estándar
col_on = next((c for c in ev.columns if c.lower() in {"onset","onset_sec"}), None)
col_du = next((c for c in ev.columns if c.lower() in {"duration","duration_sec"}), None)
col_ty = next((c for c in ev.columns if c.lower() in {"eventtype","trial_type","event_type","type"}), None)

assert col_on and col_du, "No se encontraron columnas onset/duration en el TSV original."

# (2) ECG original (Raw MNE)
raw = mne.io.read_raw_edf(src_ecg_edf, preload=True, verbose=False)
fs  = float(raw.info["sfreq"])
ecg = raw.get_data(picks=[0])[0].astype(float)
dur_total = len(ecg) / fs

print(f"\nECG original: fs = {fs:.1f} Hz | duración = {dur_total:.1f} s | muestras = {len(ecg):,}")


In [ ]:
# Celda 4: graficar ventana original con onset/offset
onset    = float(row["onset_sec"])
duration = float(row["duration_sec"])
t0, t1   = float(row["cut_start_sec"]), float(row["cut_end_sec"])  # ventana recortada en el derivado

# índices de la ventana
s0 = int(t0 * fs)
s1 = int(min(t1 * fs, len(ecg)))
t  = np.arange(s0, s1) / fs
x  = ecg[s0:s1]

plt.figure(figsize=(16,4))
plt.plot(t, x, lw=0.8, label="ECG (original, ventana recortada)")
# líneas de onset/offset
plt.axvline(onset, color="red", linestyle="--", label="onset")
plt.axvline(onset + duration, color="orange", linestyle="--", label="offset")
plt.title(f'Contexto original | t0={t0:.1f}s, t1={t1:.1f}s | evento="{row["event_type"]}"')
plt.xlabel("Tiempo (s)")
plt.ylabel("a.u.")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Celda 5: leer el segmento derivado y validar
z = np.load(dst_npz)
x_seg = z["ecg"].astype(float)
fs_seg = float(z["fs"])
dur_seg = len(x_seg) / fs_seg

with open(dst_json, "r", encoding="utf-8") as f:
    meta = json.load(f)

print("Segmento derivado:")
print("fs_seg:", fs_seg, "dur_seg:", dur_seg, "n_samples:", len(x_seg))
print("Meta.cut_start_sec:", meta["cut_start_sec"], "Meta.cut_end_sec:", meta["cut_end_sec"])

# Chequeos
ok_fs      = abs(fs_seg - fs) < 1e-6
ok_bounds  = (t0 <= onset <= t1) and (t0 <= onset + duration <= t1)
ok_durlike = abs((t1 - t0) - dur_seg) < 1.0  # tolerancia 1 s por redondeos

print("\nChequeos:")
print("fs_seg == fs original?       ", ok_fs)
print("onset/offset dentro del recorte?", ok_bounds)
print("duración segmento ≈ t1-t0?     ", ok_durlike)


In [ ]:
# Celda 6: graficar solo el segmento derivado
# Eje de tiempo relativo al inicio del segmento recortado (t = 0 en cut_start_sec)
t_seg = np.arange(len(x_seg)) / fs_seg

plt.figure(figsize=(16,4))
plt.plot(t_seg, x_seg, lw=0.8, label="ECG (segmento derivado)")
# posicionar onset relativo a t_seg=0
rel_on  = onset - t0
rel_off = onset + duration - t0
plt.axvline(rel_on, color="red", linestyle="--", label="onset (relativo)")
plt.axvline(rel_off, color="orange", linestyle="--", label="offset (relativo)")
plt.title(f'Segmento derivado (0 = cut_start = {t0:.1f}s) | evento="{row["event_type"]}"')
plt.xlabel("Tiempo relativo (s)")
plt.ylabel("a.u.")
plt.legend()
plt.tight_layout()
plt.show()
